# Deploying Café Website to AWS S3 using Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arishaprasain/S3-bucketAWS/blob/main/colab_s3_deployment.ipynb)

This notebook demonstrates how to deploy the café website to AWS S3 using Python and boto3.

## Prerequisites
- AWS Account
- AWS Access Key ID
- AWS Secret Access Key
- S3 bucket name

## Step 1: Install Required Packages

First, we need to install the AWS SDK (boto3) and AWS CLI.

In [ ]:
# Install boto3 (AWS SDK for Python) and awscli
!pip install boto3 awscli -q

## Step 2: Clone the Repository

Clone the repository containing the café website files.

In [ ]:
# Clone the repository
!git clone https://github.com/arishaprasain/S3-bucketAWS.git

# Change to the repository directory
%cd S3-bucketAWS

## Step 3: Configure AWS Credentials

**⚠️ Security Note:** For production use, consider using IAM roles or AWS Secrets Manager. Never commit credentials to version control.

You have two options:

### Option A: Direct Input (Quick but less secure)

In [ ]:
import os
from getpass import getpass

# Input your AWS credentials securely
aws_access_key_id = getpass('Enter AWS Access Key ID: ')
aws_secret_access_key = getpass('Enter AWS Secret Access Key: ')
aws_region = input('Enter AWS Region (default: us-east-1): ') or 'us-east-1'

# Set environment variables
os.environ['AWS_ACCESS_KEY_ID'] = aws_access_key_id
os.environ['AWS_SECRET_ACCESS_KEY'] = aws_secret_access_key
os.environ['AWS_DEFAULT_REGION'] = aws_region

print(f"✓ AWS credentials configured for region: {aws_region}")

### Option B: Mount Google Drive and Load Credentials from File (More Secure)

Create a file in your Google Drive (e.g., `aws_credentials.txt`) with the following format:
```
AWS_ACCESS_KEY_ID=your_access_key_here
AWS_SECRET_ACCESS_KEY=your_secret_key_here
AWS_DEFAULT_REGION=us-east-1
```

In [ ]:
# Mount Google Drive (uncomment to use)
# from google.colab import drive
# drive.mount('/content/drive')

# Load credentials from file (uncomment and update path to use)
# with open('/content/drive/MyDrive/aws_credentials.txt', 'r') as f:
#     for line in f:
#         if '=' in line:
#             key, value = line.strip().split('=', 1)
#             os.environ[key] = value
# print("✓ AWS credentials loaded from Google Drive")

## Step 4: Verify AWS Configuration

Test that AWS credentials are working by listing your S3 buckets.

In [ ]:
import boto3
from botocore.exceptions import ClientError, NoCredentialsError

try:
    s3_client = boto3.client('s3')
    response = s3_client.list_buckets()
    
    print("✓ AWS credentials verified successfully!")
    print("\nYour S3 buckets:")
    for bucket in response['Buckets']:
        print(f"  - {bucket['Name']}")
except NoCredentialsError:
    print("✗ No AWS credentials found. Please configure them in Step 3.")
except ClientError as e:
    print(f"✗ Error: {e}")

## Step 5: Create S3 Bucket (Optional)

If you don't have a bucket yet, create one. Skip this if you already have a bucket.

In [ ]:
# Create a new S3 bucket (uncomment to use)
# bucket_name = input('Enter a unique bucket name: ')
# region = os.environ.get('AWS_DEFAULT_REGION', 'us-east-1')

# try:
#     if region == 'us-east-1':
#         s3_client.create_bucket(Bucket=bucket_name)
#     else:
#         s3_client.create_bucket(
#             Bucket=bucket_name,
#             CreateBucketConfiguration={'LocationConstraint': region}
#         )
#     print(f"✓ Bucket '{bucket_name}' created successfully!")
# except ClientError as e:
#     print(f"✗ Error creating bucket: {e}")

## Step 6: Configure Your Bucket Name

Set the S3 bucket name where you want to deploy the website.

In [ ]:
# Set your S3 bucket name
BUCKET_NAME = input('Enter your S3 bucket name: ') or 'cafebucketchallengearisha'
print(f"✓ Using bucket: {BUCKET_NAME}")

## Step 7: Upload Website Files to S3

Upload all files from the `public/` directory to your S3 bucket.

In [ ]:
import os
import mimetypes
from pathlib import Path

def upload_directory_to_s3(local_directory, bucket_name, s3_prefix=''):
    """
    Upload a directory to S3 bucket with proper content types.
    """
    s3_client = boto3.client('s3')
    uploaded_files = []
    
    for root, dirs, files in os.walk(local_directory):
        # Skip .github directory
        if '.github' in root:
            continue
            
        for file in files:
            local_path = os.path.join(root, file)
            relative_path = os.path.relpath(local_path, local_directory)
            s3_path = os.path.join(s3_prefix, relative_path).replace("\\", "/")
            
            # Determine content type
            content_type, _ = mimetypes.guess_type(local_path)
            if content_type is None:
                content_type = 'binary/octet-stream'
            
            try:
                extra_args = {'ContentType': content_type}
                s3_client.upload_file(local_path, bucket_name, s3_path, ExtraArgs=extra_args)
                uploaded_files.append(s3_path)
                print(f"✓ Uploaded: {s3_path}")
            except ClientError as e:
                print(f"✗ Failed to upload {s3_path}: {e}")
    
    return uploaded_files

# Upload the public directory
print("Starting upload...\n")
uploaded = upload_directory_to_s3('public', BUCKET_NAME)
print(f"\n✓ Successfully uploaded {len(uploaded)} files to S3!")

## Step 8: Configure Bucket for Static Website Hosting

Configure the S3 bucket to serve the website as a static website.

In [ ]:
# Configure static website hosting
website_configuration = {
    'ErrorDocument': {'Key': 'error.html'},
    'IndexDocument': {'Suffix': 'index.html'},
}

try:
    s3_client.put_bucket_website(
        Bucket=BUCKET_NAME,
        WebsiteConfiguration=website_configuration
    )
    print("✓ Static website hosting configured!")
except ClientError as e:
    print(f"✗ Error configuring website: {e}")

## Step 9: Set Bucket Policy for Public Access

Make the bucket contents publicly accessible so visitors can view your website.

**⚠️ Note:** This makes all files in the bucket publicly readable.

In [ ]:
import json

# First, disable block public access settings
try:
    s3_client.delete_public_access_block(Bucket=BUCKET_NAME)
    print("✓ Public access block removed")
except ClientError as e:
    print(f"Note: {e}")

# Define bucket policy for public read access
bucket_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "PublicReadGetObject",
            "Effect": "Allow",
            "Principal": "*",
            "Action": "s3:GetObject",
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}/*"
        }
    ]
}

# Apply the bucket policy
try:
    s3_client.put_bucket_policy(
        Bucket=BUCKET_NAME,
        Policy=json.dumps(bucket_policy)
    )
    print("✓ Bucket policy applied for public access!")
except ClientError as e:
    print(f"✗ Error setting bucket policy: {e}")

## Step 10: Get Website URL

Retrieve the public URL of your deployed website.

In [ ]:
# Get the website URL
region = os.environ.get('AWS_DEFAULT_REGION', 'us-east-1')

if region == 'us-east-1':
    website_url = f"http://{BUCKET_NAME}.s3-website-us-east-1.amazonaws.com"
else:
    website_url = f"http://{BUCKET_NAME}.s3-website-{region}.amazonaws.com"

print("\n" + "="*60)
print("🎉 Deployment Complete!")
print("="*60)
print(f"\nYour website is now live at:\n{website_url}")
print("\n" + "="*60)

## Alternative: Using AWS CLI

You can also use the AWS CLI to sync files to S3 (similar to the GitHub Actions workflow).

In [ ]:
# Using AWS CLI (alternative method)
# !aws s3 sync ./public/ s3://{BUCKET_NAME} --delete
# print(f"✓ Files synced to s3://{BUCKET_NAME}")

## Bonus: List All Files in Your Bucket

In [ ]:
# List all files in the bucket
try:
    response = s3_client.list_objects_v2(Bucket=BUCKET_NAME)
    
    if 'Contents' in response:
        print(f"\nFiles in bucket '{BUCKET_NAME}':")
        print("-" * 50)
        for obj in response['Contents']:
            size_kb = obj['Size'] / 1024
            print(f"  {obj['Key']:<40} {size_kb:>8.2f} KB")
        print("-" * 50)
        print(f"Total files: {len(response['Contents'])}")
    else:
        print("No files found in bucket.")
except ClientError as e:
    print(f"✗ Error listing files: {e}")

## Cleanup: Delete All Files from Bucket (Optional)

**⚠️ Warning:** This will delete all files from your S3 bucket. Use with caution!

In [ ]:
# Uncomment to delete all files from the bucket
# def delete_all_objects(bucket_name):
#     try:
#         response = s3_client.list_objects_v2(Bucket=bucket_name)
#         if 'Contents' in response:
#             for obj in response['Contents']:
#                 s3_client.delete_object(Bucket=bucket_name, Key=obj['Key'])
#                 print(f"✓ Deleted: {obj['Key']}")
#             print(f"\n✓ All files deleted from {bucket_name}")
#         else:
#             print("Bucket is already empty.")
#     except ClientError as e:
#         print(f"✗ Error: {e}")

# delete_all_objects(BUCKET_NAME)

## Troubleshooting

### Common Issues:

1. **Access Denied Error**: Check that your AWS credentials have the necessary S3 permissions.

2. **Bucket Already Exists**: S3 bucket names must be globally unique. Try a different name.

3. **Website Not Accessible**: Ensure:
   - Static website hosting is enabled
   - Bucket policy allows public read access
   - Block public access settings are disabled

4. **Content Type Issues**: Files may not display correctly if MIME types aren't set properly. The script above handles this automatically.

### Helpful AWS CLI Commands:

```bash
# Check bucket website configuration
!aws s3api get-bucket-website --bucket {BUCKET_NAME}

# Check bucket policy
!aws s3api get-bucket-policy --bucket {BUCKET_NAME}

# List all objects
!aws s3 ls s3://{BUCKET_NAME} --recursive
```

## Additional Resources

- [AWS S3 Documentation](https://docs.aws.amazon.com/s3/)
- [Boto3 S3 Documentation](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/s3.html)
- [Hosting a Static Website on S3](https://docs.aws.amazon.com/AmazonS3/latest/userguide/WebsiteHosting.html)